# Notebook 2: Classification - Customer Churn Prediction
## ST7082CEM - Big Data Management and Data Visualisation
### Smaran Luitel | Student ID: 250087

This notebook trains and evaluates two binary classifiers to predict customer churn (`Exited`):
- **Logistic Regression** - interpretable linear baseline
- **Random Forest** - ensemble model handling non-linearity and feature interactions

Both models are evaluated using AUC-ROC, F1 Score, Precision, and Recall.
Class imbalance (20.4% positive class) is addressed via class weighting.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/user/AppData/Local/Programs/Python/Python314/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = "C:/Users/user/AppData/Local/Programs/Python/Python314/python.exe"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ST7082CEM_Classification_250087") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


## 1. Load Prepared Data

Load the raw dataset via `spark.read.csv()` and rebuild the encoding pipeline. This mirrors the preprocessing steps from Notebook 1.

In [2]:
# Load raw CSV directly with Spark and apply column renames
df_raw = (
    spark.read.csv("../dataset/Customer-Churn-Records.csv", header=True, inferSchema=True)
    .withColumnRenamed("Satisfaction Score", "SatisfactionScore")
    .withColumnRenamed("Card Type", "CardType")
    .withColumnRenamed("Point Earned", "PointEarned")
)

# Drop non-predictive and leakage columns (same as Notebook 1)
df_model = df_raw.drop("RowNumber", "CustomerId", "Surname", "Complain")
print("Loaded", df_model.count(), "rows")
print("Columns:", df_model.columns)

# Rebuild encoding pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

cat_cols = ["Geography", "Gender", "CardType"]
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
            "HasCrCard", "IsActiveMember", "EstimatedSalary",
            "SatisfactionScore", "PointEarned"]

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe") for c in cat_cols]
assembler = VectorAssembler(
    inputCols=num_cols + [c + "_ohe" for c in cat_cols],
    outputCol="features_raw", handleInvalid="keep")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
prep_model = prep_pipeline.fit(df_model)
df_prepared = prep_model.transform(df_model)
print("Pipeline applied. Ready for classification.")

Loaded 10000 rows
Pipeline applied. Ready for classification.


## 2. Train/Test Split

In [3]:
train_df, test_df = df_prepared.randomSplit([0.8, 0.2], seed=42)

print(f"Train size: {train_df.count()}")
print(f"Test size:  {test_df.count()}")

print("\nClass distribution in train set:")
train_df.groupBy("Exited").count() \
    .withColumn("pct", F.round(F.col("count") / train_df.count() * 100, 2)) \
    .orderBy("Exited").show()

print("Class distribution in test set:")
test_df.groupBy("Exited").count() \
    .withColumn("pct", F.round(F.col("count") / test_df.count() * 100, 2)) \
    .orderBy("Exited").show()

Train size: 8071


Test size:  1929

Class distribution in train set:


+------+-----+-----+
|Exited|count|  pct|
+------+-----+-----+
|     0| 6419|79.53|
|     1| 1652|20.47|
+------+-----+-----+

Class distribution in test set:


+------+-----+-----+
|Exited|count|  pct|
+------+-----+-----+
|     0| 1543|79.99|
|     1|  386|20.01|
+------+-----+-----+



## 3. Class Imbalance Strategy

The dataset has a 4.9:1 imbalance (not-churned vs churned). Without correction, models tend to predict the majority class (not-churned) and achieve high accuracy while missing most actual churners.

**Strategy:** Assign higher weight to the minority class (Exited=1) using the `weightCol` parameter in PySpark MLlib. The weight for the positive class is set to `neg/pos` to balance the effective sample sizes.

In [4]:
train_total = train_df.count()
train_pos = train_df.filter("Exited = 1").count()
train_neg = train_df.filter("Exited = 0").count()
weight_ratio = train_neg / train_pos

print(f"Train positive (churned):     {train_pos}")
print(f"Train negative (not churned): {train_neg}")
print(f"Weight ratio for minority class: {weight_ratio:.3f}")

# Add class weight column
train_weighted = train_df.withColumn(
    "classWeight",
    F.when(F.col("Exited") == 1, weight_ratio).otherwise(1.0)
)

Train positive (churned):     1652
Train negative (not churned): 6419
Weight ratio for minority class: 3.886


## 4. Logistic Regression

**Justification:** Logistic Regression is chosen as the interpretable baseline. It models the log-odds of churn as a linear combination of features, making coefficients directly interpretable. It is well-suited to binary classification and widely used in churn literature. L2 regularisation (`elasticNetParam=0.0`) is applied to prevent overfitting.

In [5]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol="Exited",
    weightCol="classWeight",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_weighted)
lr_predictions = lr_model.transform(test_df)

print("Logistic Regression model trained.")
print(f"Iterations converged: {lr_model.summary.totalIterations}")

Logistic Regression model trained.
Iterations converged: 10


### 4.1 Logistic Regression Evaluation

In [6]:
binary_eval   = BinaryClassificationEvaluator(labelCol="Exited", metricName="areaUnderROC")
mc_eval_f1    = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="f1")
mc_eval_prec  = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="weightedPrecision")
mc_eval_rec   = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="weightedRecall")
mc_eval_acc   = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="accuracy")

lr_auc  = binary_eval.evaluate(lr_predictions)
lr_f1   = mc_eval_f1.evaluate(lr_predictions)
lr_prec = mc_eval_prec.evaluate(lr_predictions)
lr_rec  = mc_eval_rec.evaluate(lr_predictions)
lr_acc  = mc_eval_acc.evaluate(lr_predictions)

print("=== Logistic Regression Results ===")
print(f"AUC-ROC:   {lr_auc:.4f}")
print(f"Accuracy:  {lr_acc:.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print(f"Precision: {lr_prec:.4f}")
print(f"Recall:    {lr_rec:.4f}")

print("\nConfusion Matrix:")
lr_predictions.groupBy("Exited", "prediction").count() \
    .orderBy("Exited", "prediction").show()

=== Logistic Regression Results ===
AUC-ROC:   0.7654
Accuracy:  0.7211
F1 Score:  0.7440
Precision: 0.7937
Recall:    0.7211

Confusion Matrix:


+------+----------+-----+
|Exited|prediction|count|
+------+----------+-----+
|     0|       0.0| 1137|
|     0|       1.0|  406|
|     1|       0.0|  132|
|     1|       1.0|  254|
+------+----------+-----+



### 4.2 Logistic Regression Coefficients (Feature Importance)

In [7]:
import pandas as pd
import numpy as np

# Feature names in the order passed to VectorAssembler
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
            "HasCrCard", "IsActiveMember", "EstimatedSalary",
            "SatisfactionScore", "PointEarned"]
cat_cols = ["Geography", "Gender", "CardType"]

coefficients = lr_model.coefficients.toArray()

# Build approximate feature name list (OHE expands categoricals)
feature_names = num_cols + [f"{c}_ohe_{i}" for c in cat_cols for i in range(3)]
feature_names = feature_names[:len(coefficients)]

coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("Top 10 features by absolute coefficient (Logistic Regression):")
print(coef_df[["feature", "coefficient"]].head(10).to_string(index=False))

Top 10 features by absolute coefficient (Logistic Regression):
        feature  coefficient
            Age     0.778222
 IsActiveMember    -0.426062
Geography_ohe_1     0.206524
        Balance     0.172493
   Gender_ohe_1     0.138280
   Gender_ohe_0    -0.138280
Geography_ohe_0    -0.137404
Geography_ohe_2    -0.079997
  NumOfProducts    -0.076398
    CreditScore    -0.068352


## 5. Random Forest

**Justification:** Random Forest is an ensemble of decision trees that handles non-linear relationships and feature interactions naturally. It is robust to outliers, does not require feature scaling, and provides feature importance scores directly. It is typically superior to Logistic Regression on tabular data with mixed feature types. `numTrees=100` provides stability; `maxDepth=10` allows complex patterns while limiting overfitting.

In [8]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="Exited",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)

print("Random Forest model trained.")
print(f"Number of trees: {rf_model.getNumTrees}")

Random Forest model trained.
Number of trees: 100


### 5.1 Random Forest Evaluation

In [9]:
rf_auc  = binary_eval.evaluate(rf_predictions)
rf_f1   = mc_eval_f1.evaluate(rf_predictions)
rf_prec = mc_eval_prec.evaluate(rf_predictions)
rf_rec  = mc_eval_rec.evaluate(rf_predictions)
rf_acc  = mc_eval_acc.evaluate(rf_predictions)

print("=== Random Forest Results ===")
print(f"AUC-ROC:   {rf_auc:.4f}")
print(f"Accuracy:  {rf_acc:.4f}")
print(f"F1 Score:  {rf_f1:.4f}")
print(f"Precision: {rf_prec:.4f}")
print(f"Recall:    {rf_rec:.4f}")

print("\nConfusion Matrix:")
rf_predictions.groupBy("Exited", "prediction").count() \
    .orderBy("Exited", "prediction").show()

=== Random Forest Results ===
AUC-ROC:   0.8633
Accuracy:  0.8688
F1 Score:  0.8531
Precision: 0.8634
Recall:    0.8688

Confusion Matrix:


+------+----------+-----+
|Exited|prediction|count|
+------+----------+-----+
|     0|       0.0| 1504|
|     0|       1.0|   39|
|     1|       0.0|  214|
|     1|       1.0|  172|
+------+----------+-----+



### 5.2 Random Forest Feature Importance

In [10]:
importances = rf_model.featureImportances.toArray()

feature_imp_df = pd.DataFrame({
    "feature": feature_names[:len(importances)],
    "importance": importances[:len(feature_names)]
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 10 features by importance (Random Forest):")
print(feature_imp_df.head(10).to_string(index=False))

# Export for Tableau
feature_imp_df.to_csv("../exports/rf_feature_importance.csv", index=False)
print("\nSaved: ../exports/rf_feature_importance.csv")

Top 10 features by importance (Random Forest):
          feature  importance
              Age    0.295531
    NumOfProducts    0.208516
          Balance    0.075053
   IsActiveMember    0.063942
      CreditScore    0.061876
      PointEarned    0.054306
  EstimatedSalary    0.054143
           Tenure    0.042507
  Geography_ohe_1    0.035003
SatisfactionScore    0.025331

Saved: ../exports/rf_feature_importance.csv


## 6. Model Comparison

In [11]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "AUC-ROC":   [round(lr_auc, 4),  round(rf_auc, 4)],
    "Accuracy":  [round(lr_acc, 4),  round(rf_acc, 4)],
    "F1 Score":  [round(lr_f1, 4),   round(rf_f1, 4)],
    "Precision": [round(lr_prec, 4), round(rf_prec, 4)],
    "Recall":    [round(lr_rec, 4),  round(rf_rec, 4)],
})

print(comparison.to_string(index=False))

# Export for Tableau
comparison.to_csv("../exports/model_comparison.csv", index=False)
print("\nSaved: ../exports/model_comparison.csv")

              Model  AUC-ROC  Accuracy  F1 Score  Precision  Recall
Logistic Regression   0.7654    0.7211    0.7440     0.7937  0.7211
      Random Forest   0.8633    0.8688    0.8531     0.8634  0.8688

Saved: ../exports/model_comparison.csv


## 7. Export Predictions for Tableau

In [12]:
# Logistic Regression predictions
lr_export = lr_predictions.select("Exited", "prediction", "probability").toPandas()
lr_export["prob_churn"] = lr_export["probability"].apply(lambda x: float(x[1]))
lr_export.drop("probability", axis=1, inplace=True)
lr_export["model"] = "LogisticRegression"
lr_export.to_csv("../exports/lr_predictions.csv", index=False)
print("Saved: ../exports/lr_predictions.csv")

# Random Forest predictions
rf_export = rf_predictions.select("Exited", "prediction", "probability").toPandas()
rf_export["prob_churn"] = rf_export["probability"].apply(lambda x: float(x[1]))
rf_export.drop("probability", axis=1, inplace=True)
rf_export["model"] = "RandomForest"
rf_export.to_csv("../exports/rf_predictions.csv", index=False)
print("Saved: ../exports/rf_predictions.csv")

Saved: ../exports/lr_predictions.csv


Saved: ../exports/rf_predictions.csv
